In [ ]:
# Modules utilisés
import pandas as pd
import requests

# Options Pandas
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option("display.max_colwidth", None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_columns', 200)


In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:145.0) Gecko/20100101 Firefox/145.0",
    "Accept-Language": "fr-FR,fr;q=0.9"
}

url = "https://fr.wikipedia.org/wiki/Liste_des_maires_des_grandes_villes_fran%C3%A7aises"
response = requests.get(url, headers=headers)
tables = pd.read_html(response.text)

C:\Users\timba\AppData\Local\Temp\ipykernel_10592\1272323389.py:8: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [34]:
df_maires_raw = tables[0].copy()

In [36]:
df_maires_V1 = df_maires_raw.drop(["Unnamed: 0", "Statut", "Département", "Sexe", "Sexe.1", "Parti (2026)", "Parti (2020)", "Parti (2014)", "Parti (2008)", "Parti (2001)", "Positionnement parti (2026)", "Positionnement parti (2020)", "Positionnement parti (2014)", "Positionnement parti (2008)", "Positionnement parti (2001)"], axis=1)

In [37]:
df_maires_V1 = df_maires_V1.rename(columns={
    "Commune" : "Ville",
    "Nom (2026)": "Nom du maire_2026",
    "Parti (2026).1": "2026",
    "Nom (2020)" : "Nom du maire_2020",
    "Parti (2020).1" : "2020",
    "Nom (2014)" : "Nom du maire_2014",
    "Parti (2014).1" : "2014",
    "Nom (2008)" : "Nom du maire_2008",
    "Parti (2008).1" : "2008",
    "Nom (2001)": "Nom du maire_2001",
    "Parti (2001).1" : "2001",
})

df_maires_V1 = df_maires_V1[["Ville", "2026", "Nom du maire_2026", "2020", "Nom du maire_2020", "2014", "Nom du maire_2014", "2008", "Nom du maire_2008", "2001", "Nom du maire_2001"]]

In [71]:
df_maires_V1.to_excel("../data/raw/historique_maires_V1.xlsx")

In [72]:
df_maires_complet = pd.read_csv("../data/raw/historique_maires_complet.csv", encoding="latin-1", sep=";")

In [ ]:
cols_annees = ["2026", "2020", "2014", "2008", "2001"]
cols_noms_maires = ["Nom du maire_2026", "Nom du maire_2020", "Nom du maire_2014", "Nom du maire_2008", "Nom du maire_2001"]

df_partis = df_maires_complet.melt(
    id_vars= "Ville",
    value_vars= cols_annees,
    var_name = "année",
    value_name="parti"
)


In [59]:
df_maires = df_maires_complet.melt(
    id_vars= "Ville",
    value_vars= cols_noms_maires,
    var_name = "année",
    value_name="Nom du maire"
)

df_maires["année"] = df_maires["année"].str.extract(r'(\d+)')

In [61]:
df_maires_final = df_partis.merge(
    df_maires,
    on=["Ville", "année"],
    how="inner"
)

In [70]:
df_maires_final.to_excel("../data/clean/historique_maires_final.xlsx")